<a href="https://colab.research.google.com/github/baanujan-18/PLC---Automation/blob/main/plc-weighted-voting-system/colab/truth_table_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PLC-Based Weighted Voting System: Truth Table Verification

This notebook verifies the truth table for a weighted voting system implemented using a Siemens S7-200 PLC.

The voting weights are:

| Shareholder | Input | Share value |
|---|---|---:|
| A | I0.0 | 1 |
| B | I0.1 | 2 |
| C | I0.2 | 2 |
| D | I0.3 | 4 |

The total vote value is calculated as:

`Total = A + 2B + 2C + 4D`

The notebook generates all 16 possible input combinations and verifies the four-bit BCD output sent to the CD4511BE decoder IC.

In [ ]:
import itertools
import pandas as pd

In [ ]:
# Create an empty list to store each truth-table row
rows = []

# Generate all possible combinations of A, B, C, and D
for A, B, C, D in itertools.product([0, 1], repeat=4):

    # Calculate the weighted vote total
    total = A + (2 * B) + (2 * C) + (4 * D)

    # ---------------------------------------------------------
    # Method 1: Obtain the required BCD bits directly from total
    # ---------------------------------------------------------
    expected_q0_0 = total & 1          # BCD bit value 1
    expected_q0_1 = (total >> 1) & 1   # BCD bit value 2
    expected_q0_2 = (total >> 2) & 1   # BCD bit value 4
    expected_q0_3 = (total >> 3) & 1   # BCD bit value 8

    # ---------------------------------------------------------
    # Method 2: Simulate the simplified PLC ladder logic
    # ---------------------------------------------------------
    q0_0 = A

    q0_1 = B ^ C            # XOR operation

    carry = B & C            # Internal memory bit M0.0

    q0_2 = D ^ carry         # XOR operation

    q0_3 = D & carry         # AND operation

    # Check whether the ladder-logic outputs match the expected BCD bits
    verified = (
        q0_0 == expected_q0_0 and
        q0_1 == expected_q0_1 and
        q0_2 == expected_q0_2 and
        q0_3 == expected_q0_3
    )

    # The original question requires the display to be blank for total = 0
    display_value = "Blank" if total == 0 else str(total)

    # Store the results
    rows.append({
        "A - I0.0": A,
        "B - I0.1": B,
        "C - I0.2": C,
        "D - I0.3": D,
        "Weighted Total": total,
        "Q0.3 - Bit 8": q0_3,
        "Q0.2 - Bit 4": q0_2,
        "Q0.1 - Bit 2": q0_1,
        "Q0.0 - Bit 1": q0_0,
        "BCD Output": f"{q0_3}{q0_2}{q0_1}{q0_0}",
        "Seven-Segment Display": display_value,
        "Logic Verification": "PASS" if verified else "FAIL"
    })

# Convert the list into a table
df = pd.DataFrame(rows)

# Display the complete truth table
df

,A - I0.0,B - I0.1,C - I0.2,D - I0.3,Weighted Total,Q0.3 - Bit 8,Q0.2 - Bit 4,Q0.1 - Bit 2,Q0.0 - Bit 1,BCD Output,Seven-Segment Display,Logic Verification
0,0,0,0,0,0,0,0,0,0,0000,Blank,PASS
1,0,0,0,1,4,0,1,0,0,0100,4,PASS
2,0,0,1,0,2,0,0,1,0,0010,2,PASS
3,0,0,1,1,6,0,1,1,0,0110,6,PASS
4,0,1,0,0,2,0,0,1,0,0010,2,PASS
5,0,1,0,1,6,0,1,1,0,0110,6,PASS
6,0,1,1,0,4,0,1,0,0,0100,4,PASS
7,0,1,1,1,8,1,0,0,0,1000,8,PASS
8,1,0,0,0,1,0,0,0,1,0001,1,PASS
9,1,0,0,1,5,0,1,0,1,0101,5,PASS


In [ ]:
# Check whether every truth-table row passed the verification
if (df["Logic Verification"] == "PASS").all():
    print("Verification successful: All 16 input combinations are correct.")
else:
    print("Verification failed: Check the rows marked as FAIL.")

Verification successful: All 16 input combinations are correct.


In [ ]:
# Save the verified truth table as a CSV file
df.to_csv("weighted_voting_truth_table.csv", index=False)

print("CSV file created successfully.")

CSV file created successfully.
